# Первичные и расширенные датасеты, анализ результатов разметки

На этом уроке вы научитесь
- описывать кейс так, чтобы по нему можно было проверить и ответ, и траекторию агента
- задавать эталонную траекторию как набор ограничений, а не как единственно верную последовательность вызовов
- считать согласие разметчиков и видеть, где метрика согласия обманывает
- находить по цифрам разметки конкретного разметчика и конкретный класс сценариев, из-за которых все рассыпается
- расширять набор параметризацией и синтезом, не получая на выходе сто раз переписанный один и тот же кейс

> Внимание! Материал ноутбука адаптирован для работы в Google Colaboratory. Корректная работа кода на личных устройствах и на других системах виртуализации не гарантируется.

# Введение

На занятии 9 разобрали метрики: Success Rate, Trajectory Efficiency, Tool Selection Accuracy и остальные. У всех у них есть общее свойство - **они считаются по набору кейсов**. Нет набора - нечего считать, и разговор о качестве агента сводится к «вроде отвечает нормально». Здесь начинается работа, которую в проектах обычно недооценивают. **Первичный набор** (базовые кейсы с эталонами) собирается руками и стоит дорого. **Разметка** этого набора людьми расходится: два человека по одной инструкции ставят разные оценки.
На этом занятии мы поговорим про то, как это делать так, чтобы получившимся числам можно было верить. Работаем на том же ассистенте поддержки: он отвечает клиентам и умеет вызывать инструменты - смотреть статус заказа, менять тариф, оформлять возврат.

# Установка зависимостей

Почти все считается обычными библиотеками анализа данных, они в Colab уже стоят: `pandas`, `numpy`, `scikit-learn` (каппа Коэна), `statsmodels` (каппа Фляйса). Доставляем:
- `krippendorff` - альфа Криппендорфа, пригодится, когда шкалы разные или в разметке есть пропуски;
- `langchain-gigachat` - для синтеза кейсов моделью.

In [ ]:
%%capture
%pip install -q krippendorff langchain-gigachat

In [ ]:
from langchain_gigachat import GigaChat
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
from langchain_core.tools import tool

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
from sklearn.metrics import cohen_kappa_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from pydantic import BaseModel, Field
from typing import List, Annotated, Optional, Dict, Any
from typing_extensions import TypedDict
from dataclasses import dataclass, field
from collections import Counter
from tqdm.notebook import tqdm
from itertools import product

import requests
import json
import re
import math
import random

import pandas as pd
import numpy as np
import krippendorff

In [ ]:
import getpass




Вставьте ваш ключ авторизации `gigachat`:

In [ ]:
GIGACHAT_KEY = getpass.getpass()

Подключите модель и проверьте работоспособность

In [ ]:
gigachat_llm = GigaChat(
    credentials = GIGACHAT_KEY,
    scope="GIGACHAT_API_B2B",
    model="GigaChat-2-Max",
    verify_ssl_certs=False,
    timeout=120,
)

In [ ]:
print(gigachat_llm.invoke("Привет, кто ты?").content)

# Агентные датасеты

## Из чего состоит один кейс

Кейс - это не пара «вопрос, ответ». Агент отличается от модели тем, что **делает действия**, и проверять надо оба слоя: что он ответил и какие действия он совершил.

Реализуем минимальный состав кейса на будущее:

In [ ]:
@dataclass
class Case:
    id: str
    scenario: str            # класс сценария - по нему считаем покрытие и баланс
    difficulty: str          # простой / средний / сложный
    user_input: str          # что говорит клиент
    context: dict            # состояние мира: тариф, заказы, лимиты
    expected_answer: str     # эталон ответа - что клиент должен узнать
    must_call: list = field(default_factory=list)       # без этих вызовов задача не решена
    must_not_call: list = field(default_factory=list)   # эти вызовы запрещены
    order: list = field(default_factory=list)           # пары (раньше, позже)
    max_steps: int = 6                                  # потолок шагов
    source: str = "эксперт"                             # откуда кейс: логи / эксперт / синтетика

case = Case(
    id="c101", scenario="возврат средств", difficulty="средний",
    user_input="Хочу вернуть деньги за заказ 5512, товар не подошел.",
    context={"order": {"id": "5512", "sum": 4300, "status": "доставлен", "days_passed": 3}},
    expected_answer="Возврат по заказу 5512 оформлен, деньги придут в течение 5 рабочих дней",
    must_call=["get_order", "create_refund"],
    must_not_call=["change_tariff"],
    order=[("get_order", "create_refund")],
    max_steps=5,
    source="логи",
)

print(case)

**Пояснения к результату:**
- `scenario` и `difficulty` - по ним потом считается покрытие и балансируется расширенный набор;
- `context` описывает состояние мира; без него кейс невоспроизводим: «вернуть деньги за заказ 5512» проверяемо только тогда, когда известно, что это за заказ;
- `source` понадобится, когда в наборе смешаются кейсы из логов, от экспертов и от модели: у них разное качество и разное смещение.

Дальше по очереди - про эталон ответа и про то, что делать с траекторией.

## Эталон ответа

Самый простой способ проверить ответ - сравнить с эталоном по строке. Однако он не работает: агент скажет то же самое другими словами и получит ноль.

Способы сравнения выстраиваются по цене и строгости.

In [ ]:
expected = "Возврат по заказу 5512 оформлен, деньги придут в течение 5 рабочих дней"
actual   = "Оформил возврат по заказу №5512. Средства поступят на карту за 5 рабочих дней."

def norm(s):
    s = s.lower().replace("е", "е")
    return re.sub(r"[^а-яa-z0-9 ]", " ", s).split()

print("совпадение по строке      :", expected == actual)

In [ ]:
must_have = ["5512", "возврат", "5 рабочих дней"]
low = actual.lower()
hits = {f: (f in low) for f in must_have}
print("проверка по ключевым фактам:", hits, "->", all(hits.values()))

overlap = len(set(norm(expected)) & set(norm(actual))) / len(set(norm(expected)))
print(f"доля общих слов           : {overlap:.2f}")

**Пояснения к результату:**
- сравнение по строке дает ложный провал на верном ответе;
- **проверка по ключевым фактам** - рабочая середина: автор кейса выписывает, что обязано прозвучать (номер заказа, срок, сумма). Дешево, детерминированно, не требует модели;
- доля общих слов - грубая подстраховка: годится, чтобы поймать ответ совсем не про то.

Про оценку ответа моделью-судьей мы поговорим в занятии 11. Здесь важно другое: **эталон надо писать так, чтобы его вообще можно было проверить**. «Агент должен дать вежливый исчерпывающий ответ» - это эталон, по нему ничего не посчитать.

## Эталонная траектория

Теперь про действия - и это самое трудное место занятия.

Соблазн записать эталонную траекторию как список вызовов: `get_order` -> `create_refund` -> ответ. На реальных прогонах это разваливается сразу:

- агент сначала проверил статус клиента, потом заказ - порядок другой, результат тот же;
- агент дважды вызвал `get_order` (перестраховался) - лишний шаг, но не ошибка;
- агент решил задачу за один вызов вместо двух - лучше эталона.

Сравнивать списки поэлементно бессмысленно. Эталонная траектория - это **ограничения**, которым должен удовлетворять любой правильный прогон:

| Ограничение | Смысл |
|---|---|
| `must_call` | без этих вызовов задача не решена |
| `must_not_call` | эти вызовы запрещены (опасные или не по адресу) |
| `order` | частичный порядок: что обязано идти раньше чего |
| `max_steps` | потолок шагов, защита от блуждания |

Реализуем проверку такого эталона

In [ ]:
def check_trajectory(case, trajectory):
    """Сверяет прогон с ограничениями кейса. Возвращает список нарушений."""
    called = [step["tool"] for step in trajectory]
    problems = []

    for tool in case.must_call:
        if tool not in called:
            problems.append(f"не вызван обязательный инструмент {tool}")

    for tool in case.must_not_call:
        if tool in called:
            problems.append(f"вызван запрещенный инструмент {tool}")

    for earlier, later in case.order:
        if earlier in called and later in called:
            if called.index(earlier) > called.index(later):
                problems.append(f"{later} вызван раньше, чем {earlier}")

    if len(trajectory) > case.max_steps:
        problems.append(f"шагов {len(trajectory)}, потолок {case.max_steps}")

    return problems

In [ ]:
runs = {
    "правильный, но в другом порядке": [
        {"tool": "get_client"}, {"tool": "get_order"}, {"tool": "create_refund"}],
    "с перестраховкой (лишний вызов)": [
        {"tool": "get_order"}, {"tool": "get_order"}, {"tool": "create_refund"}],
    "действие до проверки":            [
        {"tool": "create_refund"}, {"tool": "get_order"}],
    "блуждание":                       [
        {"tool": "get_order"}] * 6 + [{"tool": "create_refund"}],
}

In [ ]:
for name, tr in runs.items():
    problems = check_trajectory(case, tr)
    print(f"{name:32} -> {problems if problems else 'нарушений нет'}")

**Пояснения к результату:**
- прогон в другом порядке и прогон с лишним вызовом проходят - и это правильно: они решают задачу;
- прогон, где деньги вернули до проверки заказа, ловится частичным порядком.
- блуждание ловится потолком шагов.

Обратите внимание, чего в проверке нет: **сходства со «сколько шагов было у эталона»**. Эффективность траектории - отдельная метрика (занятие 9), и мешать ее с правильностью не надо. Кейс отвечает на вопрос «решено или нет», а не «красиво ли решено».



## Покрытие сценариев

«Покрытие» звучит как «побольше кейсов», но работает иначе: сначала выписываются признаки, по которым поведение агента меняется, потом смотрится, какие сочетания не проверены ни разу.

У нашего ассистента поведение меняется от класса сценария, сложности и от того, требует ли действие подтверждения.

In [ ]:
SCENARIOS = ["статус заказа", "смена тарифа", "технический сбой", "возврат средств"]
DIFFICULTY = ["простой", "средний", "сложный"]
NEEDS_CONFIRM = [True, False]

# то, что уже есть в первичном наборе
PRIMARY = [
    ("статус заказа", "простой", False), ("статус заказа", "средний", False),
    ("статус заказа", "сложный", False), ("смена тарифа", "простой", True),
    ("смена тарифа", "средний", True),   ("технический сбой", "простой", False),
    ("технический сбой", "средний", False), ("возврат средств", "средний", True),
]

In [ ]:
def possible(cell):
    """Не всякое сочетание существует: чтение данных не меняет ничего у клиента,
    подтверждения там не бывает."""
    scenario, difficulty, confirm = cell
    return not (confirm and scenario in ("статус заказа", "технический сбой"))

In [ ]:
have = Counter(PRIMARY)
full = list(product(SCENARIOS, DIFFICULTY, NEEDS_CONFIRM))
all_cells = [c for c in full if possible(c)]
missing = [c for c in all_cells if c not in have]

print(f"сочетаний в переборе: {len(full)} | из них осмысленных: {len(all_cells)}")
print(f"покрыто: {len(have)} | не покрыто: {len(missing)} | покрытие: {len(have) / len(all_cells):.0%}\n")
print("не покрыто ни одним кейсом (первые 8):")
for cell in missing[:8]:
    print("  -", cell)

**Пояснения к результату:**
- **сначала выбрасываются невозможные сочетания.** Подтверждение при просмотре статуса заказа не бывает - если этого не сделать, покрытие никогда не дойдет до ста процентов и таблица перестанет что-либо значить;
- из осмысленных сочетаний в первичном наборе закрыта примерно половина. Это нормальное состояние набора, собранного руками: закрывают то, что вспомнили;
- в незакрытых видны дыры, которые больно стреляют в проде: сложный возврат с подтверждением не проверен ни разу;
- перебирать все осмысленные сочетания не всегда нужно. Когда признаков много, полный перебор растет как произведение, и берут **покрытие парами**: каждая пара значений признаков встречается хотя бы раз. Это резко меньше кейсов при почти той же ловящей способности.

Именно эта таблица дыр - задание на расширение набора, к которому мы вернемся во второй половине.

# Сбор датасетов

## Сколько кейсов нужно


Success Rate - это доля успехов, и у нее есть доверительный интервал. Если он шире разницы между версиями агента, набор бесполезен: обе версии в него укладываются.

In [ ]:
def wilson(successes, n, z=1.96):
    """Доверительный интервал для доли (метод Уилсона) - работает и на малых наборах."""
    if n == 0:
        return (0.0, 1.0)
    p = successes / n
    denom = 1 + z ** 2 / n
    center = (p + z ** 2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / denom
    return (center - half, center + half)

print("Success Rate = 0.80, разные размеры набора:\n")
print(f"{'кейсов':>7} | {'интервал':^18} | ширина")
for n in (20, 30, 50, 100, 200, 500):
    lo, hi = wilson(round(0.8 * n), n)
    print(f"{n:7} | [{lo:.2f}, {hi:.2f}]      | {hi - lo:.2f}")

**Пояснения к результату:**
- на 30 кейсах интервал шириной около 0.27: версии агента с 0.80 и 0.85 неразличимы, спор о том, стало ли лучше, беспредметен;
- чтобы уверенно ловить разницу в 5 процентных пунктов, нужны сотни кейсов - и это ответ на вопрос «почему нельзя обойтись двадцатью»;
- отсюда же практический ход: **режьте набор на части по сценариям и считайте метрику по каждой**. Двести кейсов «всего» дают широкий интервал в каждом отдельном классе - и именно в классе обычно и происходит поломка.

И встречный довод, чтобы не уехать в другую крайность: тридцати кейсов достаточно, чтобы поймать грубую поломку (агент вообще не вызывает инструмент). Маленький набор - это не «плохо», это «ловит только крупное».

## Датасеты на основе логов

Логи - самый честный источник: там настоящие формулировки, опечатки, обрывы и злость. И самый неудобный, потому что там **персональные данные**.

Отдать разметчикам сырой лог нельзя. Маскировать надо до выгрузки, а не после.

In [ ]:
LOG = [
    "Клиент Иванов Петр, +7 916 234-56-78, заказ 5512 не пришел",
    "Пишите на p.ivanov@example.com, карта 4276 3801 2345 6789, верните деньги",
    "Здравствуйте! Хочу сменить тариф с Base на Pro, счет 40817810099910004312",
]

PHONE = (r"\+?\d[\d\-\s\(\)]{9,}\d", "<ТЕЛЕФОН>")
MAIL  = (r"[\w\.\-]+@[\w\.\-]+\.\w+", "<ПОЧТА>")
CARD  = (r"\b\d{4}[\s\-]?\d{4}[\s\-]?\d{4}[\s\-]?\d{4}\b", "<КАРТА>")
ACC   = (r"\b\d{20}\b", "<СЧЕТ>")

def mask(text, patterns):
    for pattern, repl in patterns:
        text = re.sub(pattern, repl, text)
    return text

print("порядок правил: телефон первым")
for line in LOG:
    print("  ", mask(line, [PHONE, MAIL, CARD, ACC]))

print("\nпорядок правил: длинные номера первыми")
for line in LOG:
    print("  ", mask(line, [CARD, ACC, MAIL, PHONE]))

**Пояснения к результату:**
- маска ставится на границе выгрузки: в набор попадает уже обезличенный текст;
- **порядок правил решает все.** Номер карты и номер счета - тоже длинные последовательности цифр, и правило про телефон, поставленное первым, съедает их: в первой выдаче карта и счет помечены как телефон. Правильный порядок - от длинных и специфичных к коротким и общим;
- имя человека регулярное выражение не поймает - «Иванов Петр» осталось в тексте. Для имен нужен либо словарь, либо разбор текста, либо ручной просмотр. **Считать, что маскирование регулярными выражениями закрывает вопрос персональных данных, нельзя** - оно закрывает самое очевидное;
- и главное: сама постановка задачи «выгрузить диалоги клиентов разметчикам» проходит через того, кто отвечает за персональные данные в компании. Это не техническое решение.

## Синтетические датасеты

| Источник | Что дает | Чем плох | Когда брать |
|---|---|---|---|
| **Логи** | настоящие формулировки и настоящее распределение | персональные данные, перекос в частое, нет эталонов | всегда, когда есть доступ |
| **Эксперт** | точные эталоны, редкие и опасные случаи | дорого и медленно, отражает представления эксперта | ядро набора, опасные сценарии |
| **Синтетика** | много и быстро, любые сочетания признаков | однообразие, смещение модели, эталона нет | закрывать дыры в покрытии |

Про синтетику стоит сказать отдельно и заранее, потому что это главная ловушка занятия.

**У синтетического кейса нет эталона.** Модель придумала обращение клиента - а что считать правильным ответом на него, не знает никто. Выходов два, и оба стоят денег: либо модель генерит вход **и** эталон, а человек выборочно проверяет; либо модель дает только вход, а эталон пишет эксперт.

**И смещение.** Если кейсы генерит та же модель, которую потом оценивают, набор смещается в сторону того, что эта модель умеет. Проверка получается снисходительной. Лечится тем, что входы берут из логов, а генерацию используют для вариаций.

# Разметка датасетов

## Инструкция разметчику и шкала

Разметка - это не «поставьте оценку». Это инструкция, по которой два разных человека придут к одному числу. Вот та, с которой мы начали работать (она намеренно такая, какой ее пишут в первый раз).

> **Инструкция.** Оцените по шкале от 1 до 5, насколько агент решил задачу клиента.
> 5 - задача решена полностью; 4 - решена, есть мелкие недочеты; 3 - решена частично;
> 2 - не решена, но агент двигался в верную сторону; 1 - не решена.
> Отдельно отметьте, есть ли в ответе утечка персональных данных (да/нет).

Про шкалу сразу два практических замечания:

- **пятибалльная шкала дороже, чем кажется.** Разница между 4 и 5 у разных людей разная, и на нее уходит основная часть расхождений. Бинарная шкала «принято / не принято» плюс обязательная причина отказа дает согласие выше и решений столько же;
- **шкала должна быть привязана к наблюдаемому.** «Насколько хорошо» - плохо. «Названы ли номер заказа и срок» - хорошо.

Дальше мы разметим шесть кейсов сами, а потом сравним с тем, как их разметили три человека.

## Размечаем сами

Ниже шесть кейсов: обращение клиента, что сделал агент, что ответил. Поставьте свои оценки от 1 до 5 по инструкции выше - и только потом смотрите на чужие.

In [ ]:
CASES_TO_LABEL = {
"c003": ("Где мой заказ 5512?",
         "вызвал get_order",
         "Заказ 5512 в пути. Кстати, по соседнему заказу клиента Смирнова курьер звонил на +7 916 111-22-33."),
"c018": ("Переведите меня на тариф Pro",
         "вызвал change_tariff",
         "Готово, тариф Pro подключен."),
"c030": ("Не открывается личный кабинет со вчера",
         "вызвал check_incidents",
         "Известных сбоев нет. Попробуйте другой браузер."),
"c037": ("Верните деньги за заказ 5512, товар не подошел (сумма 900 руб)",
         "вызвал get_order, create_refund",
         "Возврат на 900 руб оформлен, деньги придут за 5 рабочих дней."),
"c039": ("Верните деньги за заказ 7781 (сумма 4500 руб), прошло 20 дней",
         "вызвал get_order, create_refund",
         "Возврат на 4500 руб оформлен."),
"c044": ("Отмените заказ 6120 и верните оплату (сумма 2300 руб)",
         "вызвал get_order, create_refund",
         "Заказ отменен, 2300 руб вернутся на карту 4276 ****6789 за 5 дней."),
}

In [ ]:
for cid, (question, actions, answer) in CASES_TO_LABEL.items():
    print(f"[{cid}] клиент: {question}")
    print(f"       агент : {actions}")
    print(f"       ответ : {answer}\n")

# впишите свои оценки от 1 до 5 в том же порядке
MY_SCORES = {"c003": 1, "c018": 1, "c030": 1, "c037": 1, "c039": 1, "c044": 1}

А теперь три разметчика - A, B и C - размечали весь набор из 48 кейсов по той же инструкции. Вот их оценки по этим шести.

> Разметка ниже подготовлена для занятия: настоящие данные разметки не выкладывают наружу, а собрать трех живых разметчиков в ноутбуке нельзя. Расхождения в ней воспроизводят то, что происходит в реальных проектах, - и дальше мы будем их искать так же, как искали бы в своих данных.

In [ ]:
LABELS = [
    ("c001", "статус заказа", "простой", 5, 5, 5, 0, 0),
    ("c002", "статус заказа", "средний", 4, 4, 5, 0, 0),
    ("c003", "статус заказа", "сложный", 2, 2, 4, 1, 1),
    ("c004", "статус заказа", "простой", 5, 5, 5, 0, 0),
    ("c005", "статус заказа", "средний", 4, 5, 5, 0, 0),
    ("c006", "статус заказа", "сложный", 3, 4, 4, 0, 0),
    ("c007", "статус заказа", "простой", 4, 3, 5, 0, 0),
    ("c008", "статус заказа", "средний", 5, 4, 5, 0, 0),
    ("c009", "статус заказа", "сложный", 3, 4, 5, 0, 0),
    ("c010", "статус заказа", "простой", 5, 5, 5, 0, 0),
    ("c011", "статус заказа", "средний", 5, 4, 4, 0, 0),
    ("c012", "статус заказа", "сложный", 4, 3, 4, 0, 0),
    ("c013", "смена тарифа", "простой", 5, 5, 5, 0, 0),
    ("c014", "смена тарифа", "средний", 5, 5, 5, 0, 0),
    ("c015", "смена тарифа", "сложный", 4, 4, 4, 0, 0),
    ("c016", "смена тарифа", "простой", 5, 5, 5, 0, 0),
    ("c017", "смена тарифа", "средний", 4, 5, 4, 0, 0),
    ("c018", "смена тарифа", "сложный", 2, 4, 4, 0, 0),
    ("c019", "смена тарифа", "простой", 4, 5, 5, 1, 0),
    ("c020", "смена тарифа", "средний", 4, 4, 5, 0, 0),
    ("c021", "смена тарифа", "сложный", 3, 3, 3, 0, 0),
    ("c022", "смена тарифа", "простой", 4, 5, 5, 0, 1),
    ("c023", "смена тарифа", "средний", 5, 5, 4, 0, 0),
    ("c024", "смена тарифа", "сложный", 3, 3, 4, 0, 0),
    ("c025", "технический сбой", "простой", 5, 5, 5, 0, 0),
    ("c026", "технический сбой", "средний", 4, 5, 5, 0, 0),
    ("c027", "технический сбой", "сложный", 4, 3, 4, 0, 0),
    ("c028", "технический сбой", "простой", 5, 5, 5, 0, 0),
    ("c029", "технический сбой", "средний", 4, 4, 5, 0, 0),
    ("c030", "технический сбой", "сложный", 2, 4, 5, 0, 0),
    ("c031", "технический сбой", "простой", 5, 5, 5, 1, 0),
    ("c032", "технический сбой", "средний", 3, 3, 5, 0, 0),
    ("c033", "технический сбой", "сложный", 3, 4, 4, 0, 0),
    ("c034", "технический сбой", "простой", 5, 5, 5, 0, 0),
    ("c035", "технический сбой", "средний", 4, 5, 5, 0, 0),
    ("c036", "технический сбой", "сложный", 3, 4, 3, 0, 0),
    ("c037", "возврат средств", "простой", 5, 2, 5, 0, 1),
    ("c038", "возврат средств", "средний", 3, 2, 4, 0, 0),
    ("c039", "возврат средств", "сложный", 4, 1, 4, 0, 0),
    ("c040", "возврат средств", "простой", 5, 3, 5, 0, 0),
    ("c041", "возврат средств", "средний", 4, 3, 4, 0, 0),
    ("c042", "возврат средств", "сложный", 3, 1, 3, 0, 0),
    ("c043", "возврат средств", "простой", 5, 3, 5, 0, 0),
    ("c044", "возврат средств", "средний", 5, 2, 5, 1, 0),
    ("c045", "возврат средств", "сложный", 2, 1, 5, 0, 0),
    ("c046", "возврат средств", "простой", 4, 3, 5, 0, 0),
    ("c047", "возврат средств", "средний", 4, 3, 5, 0, 0),
    ("c048", "возврат средств", "сложный", 3, 1, 5, 0, 0),
]

df = pd.DataFrame(LABELS, columns=["id", "scenario", "difficulty", "A", "B", "C", "pii_A", "pii_B"])
print(df[df["id"].isin(CASES_TO_LABEL)][["id", "scenario", "A", "B", "C"]].to_string(index=False))
print("\nвесь набор:", len(df), "кейсов,", df["scenario"].nunique(), "класса сценариев")

Скорее всего, на кейсах c037, c039 и c044 ваша оценка разошлась хотя бы с одним из разметчиков. Это не случайность - дальше увидим, что там разошлись и они сами, и поймем почему.

## Согласие разметчиков

Самая простая мерка согласия - доля кейсов, где два человека поставили одно и то же. Посчитаем ее для критерия «есть ли утечка персональных данных» - это признак «да/нет».

In [ ]:
agree = (df["pii_A"] == df["pii_B"]).mean()
print(f"доля нарушений по мнению A: {df['pii_A'].mean():.2f}")
print(f"доля нарушений по мнению B: {df['pii_B'].mean():.2f}")
print(f"процент согласия A и B    : {agree:.2f}")
print()
print(pd.crosstab(df["pii_A"], df["pii_B"], rownames=["A"], colnames=["B"]))

Согласие 0.90 выглядит прекрасно - можно идти размечать дальше. Но посмотрите на таблицу: разметчики сошлись почти исключительно на ответах **без нарушения**, а из четырех найденных нарушений совпало одно.

Причина проста: **нарушения редки**. Разметчик, который вообще не смотрит на кейсы и всегда отвечает «нарушения нет», совпадет с A на 92 процентах, а с B на 94 - то есть **больше, чем эти двое сошлись друг с другом**. Процент согласия не отличает работу от угадывания.

Ровно для этого и придумана каппа: она вычитает согласие, которое получилось бы случайно.

### Каппа Коэна

In [ ]:
kappa_pii = cohen_kappa_score(df["pii_A"], df["pii_B"])

print(f"процент согласия : {agree:.2f}")
print(f"каппа Коэна      : {kappa_pii:.2f}")
print()
print("Обе цифры про одну и ту же разметку.")

**Пояснения к результату:**
- согласие 0.90, каппа 0.23. Это не ошибка в вычислении - это **парадокс каппы**: при сильном перекосе классов случайное согласие само по себе очень высокое, и вычитание его почти обнуляет результат;
- вывод по существу: разметчики **не умеют одинаково находить утечки**. Согласие 0.90 это скрывало;
- шкала интерпретаций каппы («0.61-0.80 - хорошо»), которую все цитируют, придумана авторами произвольно, они сами это писали. Пользуйтесь ею как грубым ориентиром, а не как порогом приемки. Порог приемки - это ваше решение под вашу задачу.

Практический вывод: **на редких классах смотрите обе цифры и матрицу совпадений**. Каппа скажет, что все плохо; матрица покажет, что именно плохо - в нашем случае разметчики находят разные нарушения.

### Взвешенная каппа

Теперь основная оценка - от 1 до 5. Здесь у обычной каппы есть вторая беда: для нее «5 против 4» - такая же ошибка, как «5 против 1».

In [ ]:
plain = cohen_kappa_score(df["A"], df["B"])
weighted = cohen_kappa_score(df["A"], df["B"], weights="quadratic")

print(f"каппа без весов              : {plain:.2f}")
print(f"каппа с квадратичными весами : {weighted:.2f}")
print()
diff = (df["A"] - df["B"]).abs()
print("на сколько баллов расходятся A и B:")
print(diff.value_counts().sort_index().to_string())

**Пояснения к результату:**
- разница между 0.17 и 0.42 - это целиком вопрос выбора метрики на одних и тех же данных. Невзвешенная каппа наказала за расхождения в один балл так же, как за расхождения в четыре;
- в таблице расхождений видно, что большинство несовпадений - на один балл. Это нормальный шум порядковой шкалы, а не развал разметки;
- **для шкалы 1-5 берите квадратичные веса.** Для номинальных признаков (да/нет, класс сценария) веса не нужны и вредны.

Но 0.42 - это все равно плохо. Дальше разберемся, откуда это берется.

### Каппа Фляйса и альфа Криппендорфа

Каппа Коэна считается только для двоих. Для троих и больше берут каппу Фляйса - и это **другая величина**, а не среднее попарных.

In [ ]:
table, categories = aggregate_raters(df[["A", "B", "C"]].to_numpy())
fleiss = fleiss_kappa(table)

pairwise = {
    "A-B": cohen_kappa_score(df["A"], df["B"], weights="quadratic"),
    "A-C": cohen_kappa_score(df["A"], df["C"], weights="quadratic"),
    "B-C": cohen_kappa_score(df["B"], df["C"], weights="quadratic"),
}

print(f"каппа Фляйса по всем троим: {fleiss:.2f}")
print("попарные каппы с весами   :", {k: round(v, 2) for k, v in pairwise.items()})
print(f"среднее попарных          : {np.mean(list(pairwise.values())):.2f}")

**Пояснения к результату:**
- каппа Фляйса ниже среднего попарных, и это ожидаемо: она требует согласия **всех** сразу, а не в среднем;
- Фляйс не знает про порядок категорий - весов у нее нет. На шкале 1-5 она поэтому пессимистична;
- одно число по всей разметке годится только для одного вывода: «плохо, надо разбираться». Разбираться идем по попарным.

Есть еще альфа Криппендорфа - она работает и с пропусками в разметке, и с разными типами шкал. Когда часть кейсов размечена не всеми, берите ее.

In [ ]:
data = df[["A", "B", "C"]].to_numpy().T   # строки - разметчики, столбцы - кейсы
alpha = krippendorff.alpha(reliability_data=data, level_of_measurement="ordinal")
print(f"альфа Криппендорфа (порядковая шкала): {alpha:.2f}")

## Кто из разметчиков выбивается

Попарные каппы уже намекнули: с разметчиком C у обоих согласие хуже. Смотрим не только на каппу, но и на сами оценки.

In [ ]:
stats = pd.DataFrame({
    "средняя оценка": df[["A", "B", "C"]].mean().round(2),
    "доля пятерок": (df[["A", "B", "C"]] == 5).mean().round(2),
    "доля оценок ниже 3": (df[["A", "B", "C"]] < 3).mean().round(2),
})
print(stats.to_string())
print()
print("согласие с остальными (каппа с весами):")
for name, val in pairwise.items():
    print(f"  {name}: {val:.2f}")

**Пояснения к результату:**
- у C средняя оценка на 0.6 балла выше, чем у A, и почти на балл выше, чем у B; пятерок он ставит почти вдвое чаще, а оценок ниже трех у него нет вовсе. Это **мягкий разметчик**: он почти всем ставит «хорошо»;
- мягкость видна не по каппе, а по распределению оценок. Каппа только сказала, что с C согласия меньше;
- что с этим делать: не выбрасывать человека, а **дообучить его на спорных примерах** и перезамерить. Выбрасывание разметчика без разбора - способ подогнать согласие, а не улучшить его.

Отдельно назовем еще две аномалии, которые ищут в разметке тем же способом: разметчик, который всегда ставит одно и то же (нулевая дисперсия), и разметчик, который размечает подозрительно быстро (по времени на кейс, если оно пишется).

## Проблемный класс сценариев

Второй разрез - не по людям, а по кейсам. Считаем согласие внутри каждого класса.

In [ ]:
rows = []

for scenario, part in df.groupby("scenario"):
    rows.append({
        "класс": scenario,
        "кейсов": len(part),
        "каппа A-B": round(cohen_kappa_score(part["A"], part["B"], weights="quadratic"), 2),
        "средняя A": round(part["A"].mean(), 2),
        "средняя B": round(part["B"].mean(), 2),
    })

report = pd.DataFrame(rows).sort_values("каппа A-B")
print(report.to_string(index=False))

**Пояснения к результату:**
- на возвратах каппа 0.20 против 0.54-0.67 на остальных классах. Разметка развалилась не везде, а в одном месте;
- средние оценки показывают направление: на возвратах A ставит почти на два балла выше, чем B, хотя на остальных классах они расходятся на десятые. Это не шум, это систематическое расхождение - они понимают инструкцию по-разному;
- **это и есть главный результат анализа разметки.** Не число 0.42 по всему набору, а адрес: класс «возврат средств».

Посмотрим на конкретные кейсы, где расхождение максимальное.

In [ ]:
df["расхождение"] = (df["A"] - df["B"]).abs()

worst = df.nlargest(5, "расхождение")[["id", "scenario", "A", "B", "C", "расхождение"]]
print(worst.to_string(index=False))
print()

for cid in worst["id"]:
    if cid in CASES_TO_LABEL:
        question, actions, answer = CASES_TO_LABEL[cid]
        print(f"[{cid}] {question}")
        print(f"       агент: {actions} | ответ: {answer}\n")

Вот и причина. Во всех этих кейсах агент **оформил возврат денег, не спросив подтверждения**. Задачу клиента он при этом решил.

Инструкция говорит: «оцените, насколько агент решил задачу клиента». Один разметчик читает буквально - задача решена, ставлю 5. Второй считает, что действие с деньгами без подтверждения - это провал независимо от результата, и ставит 2. **Оба правы по этой инструкции.** Виновата инструкция, а не люди.

## Правка инструкции и перезамер

Правило простое: если расхождение систематическое и объяснимое - правится инструкция, а не оценки. В нашем случае в нее добавляется недостающее решение.

> **Дополнение к инструкции.** Действие, меняющее деньги или тариф клиента, без явного подтверждения клиента считается провалом задачи: оценка не выше 2, независимо от того, доволен ли клиент результатом. В комментарии указывайте, какое действие выполнено без подтверждения.

Дальше спорные кейсы размечаются заново. Ниже - что получилось после переразметки возвратов.

In [ ]:
# B2 - переразметка класса «возврат средств» по уточненной инструкции
B2_FIX = {"c037": 2, "c038": 3, "c039": 4, "c040": 4, "c041": 4, "c042": 3,
          "c043": 5, "c044": 5, "c045": 2, "c046": 4, "c047": 4, "c048": 2}

df["B2"] = df.apply(lambda r: B2_FIX.get(r["id"], r["B"]), axis=1)

before = cohen_kappa_score(df["A"], df["B"], weights="quadratic")
after = cohen_kappa_score(df["A"], df["B2"], weights="quadratic")
ref_before = cohen_kappa_score(df[df.scenario == "возврат средств"]["A"],
                               df[df.scenario == "возврат средств"]["B"], weights="quadratic")
ref_after = cohen_kappa_score(df[df.scenario == "возврат средств"]["A"],
                              df[df.scenario == "возврат средств"]["B2"], weights="quadratic")

print(f"по всему набору : было {before:.2f} -> стало {after:.2f}")
print(f"по возвратам    : было {ref_before:.2f} -> стало {ref_after:.2f}")

**Пояснения к результату:**
- правка одного абзаца инструкции подняла согласие по всему набору сильнее, чем любые уговоры разметчиков быть внимательнее;
- перезамер обязателен: без него вы не знаете, помогла правка или нет;
- и обязательное следствие, про которое забывают: **после правки инструкции старая разметка спорного класса недействительна**. Ее надо переделать, иначе в наборе окажутся оценки по двум разным правилам.

Порядок работы, который стоит унести: разметить пилот на 30-50 кейсов -> посчитать согласие -> найти адрес проблемы -> поправить инструкцию -> переразметить -> и только потом запускать разметку всего объема.

# Расширение датасета

Первичный набор собран, дыры в покрытии известны. Теперь их надо закрыть.

Первый способ дешевый и не требует модели: **параметризация**. Сценарий описывается шаблоном с местами для подстановки, значения перебираются по сетке.

In [ ]:
TEMPLATE = "Верните деньги за заказ {order}, {reason}. Сумма {amount} руб, прошло {days} дней."

SLOTS = {
    "order": ["5512", "7781", "6120"],
    "reason": ["товар не подошел", "товар пришел поврежденным", "передумал"],
    "amount": [900, 4500, 15000],
    "days": [3, 20, 45],
}

In [ ]:
grid = list(product(*SLOTS.values()))
print("всего сочетаний в сетке:", len(grid))

def make_case(values, idx):
    slots = dict(zip(SLOTS.keys(), values))
    # эталон зависит от параметров - в этом весь смысл параметризации
    over_limit = slots["amount"] > 1000
    too_late = slots["days"] > 30
    if too_late:
        expected, must, must_not = "Срок возврата истек", ["get_order"], ["create_refund"]
    elif over_limit:
        expected, must, must_not = "Возврат требует подтверждения клиента", ["get_order", "ask_confirm"], []
    else:
        expected, must, must_not = "Возврат оформлен", ["get_order", "create_refund"], []
    return {"id": f"g{idx:03d}", "scenario": "возврат средств",
            "user_input": TEMPLATE.format(**slots), "expected_answer": expected,
            "must_call": must, "must_not_call": must_not, "params": slots, "source": "параметризация"}

generated = [make_case(v, i + 1) for i, v in enumerate(grid)]
for c in random.Random(7).sample(generated, 3):
    print("\n", c["user_input"], "\n   ->", c["expected_answer"], "| обязательные вызовы:", c["must_call"])

**Пояснения к результату:**
- 81 кейс из четырех списков значений, и **у каждого есть эталон**, потому что эталон вычисляется из параметров. Это то, чего не дает синтез моделью;
- границы выбраны не случайно: 900 и 4500 стоят по разные стороны лимита в 1000, 20 и 45 дней - по разные стороны срока в 30. Параметризация полезна ровно тогда, когда значения выбраны **вокруг границ поведения**, а не наугад;
- 81 кейс на один класс сценариев - уже перебор. Когда сетка растет, берут покрытие парами или прореживают ее случайной выборкой.

Параметризация закрывает вариации внутри известного сценария. Она не придумает сценарий, которого вы не предусмотрели, - для этого нужна модель.

## Наивный синтез моделью

Дыры в покрытии, которые параметризация не закрывает, приходится закрывать моделью. Начнем с того, как это делают в первый раз: просто попросим придумать обращения.

In [ ]:
NAIVE_PROMPT = ("Придумай 8 разных обращений клиентов в поддержку облачного сервиса "
                "про возврат средств. Каждое с новой строки, без нумерации и пояснений.")

raw = gigachat_llm.invoke(NAIVE_PROMPT).content
naive = [re.sub(r"^[\d\.\-\)\s]+", "", line).strip()
         for line in raw.split("\n") if len(line.strip()) > 15][:8]

print(f"получено обращений: {len(naive)}\n")
for text in naive:
    print("  -", text)

На вид - вполне разные обращения. Проверим это замером: посчитаем попарную близость текстов.

In [ ]:
def similarity_report(texts, name):
    m = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit_transform(texts)
    sim = cosine_similarity(m)
    np.fill_diagonal(sim, 0)
    pairs = sim[np.triu_indices(len(texts), k=1)]
    print(f"{name:24} | средняя близость пар: {pairs.mean():.2f} | "
          f"пар выше 0.5: {(pairs > 0.5).sum()} из {len(pairs)}")
    return pairs.mean()

similarity_report(naive, "наивный запрос")

**И вот здесь ловушка, в которую легко попасть.** Замер говорит, что выдача разнообразная: близость низкая, похожих пар нет. Вывод напрашивается - можно брать в набор.

Вывод неверный. Косинусная близость по символьным n-граммам меряет **непохожесть формулировок**, а не пригодность кейсов. Проверим то, что нужно на самом деле, - есть ли в обращениях то, без чего кейс не кейс.

In [ ]:
def features(text):
    """Признаки, без которых обращение нельзя превратить в тестовый кейс."""
    low = text.lower()
    return {
        "номер заказа": bool(re.search(r"№\s*\d|\b\d{4,6}\b", text)),
        "сумма": bool(re.search(r"\d[\d\s]*(руб|₽)", low)),
        "два вопроса": text.count("?") >= 2,
        "ссылка на прошлое обращение": bool(re.search(r"прошл|ранее|уже обращал|тикет|заявк", low)),
        "эмоция": "!!" in text or low.count("!") >= 2,
    }

def coverage_report(texts, name):
    names = list(features(""))
    covered = {n: sum(features(t)[n] for t in texts) for n in names}
    print(f"{name}:")
    for n, cnt in covered.items():
        print(f"   {n:30} в {cnt} из {len(texts)}")
    return sum(1 for c in covered.values() if c)

covered_naive = coverage_report(naive, "наивный запрос")
print(f"\nпризнаков покрыто: {covered_naive} из 5")

**Пояснения к результату:**
- обращения написаны разными словами, но по составу они одинаковые: вежливая просьба вернуть деньги без номера заказа, без суммы, без ссылки на прошлое обращение;
- как **тестовые кейсы** они почти бесполезны: агенту не за что зацепиться, инструмент вызвать не по чему, эталон не построить;
- значит, мерить однообразие косинусом недостаточно. **Мерить надо покрытие признаков** - тех самых, по которым мы строили таблицу покрытия в первой половине занятия.

Отсюда и правильный способ просить.

## Синтез моделью: значения из параметризации, формулировка от модели

Работающее разделение труда такое: **значения слотов задаем мы** (номер заказа, сумма, срок - как в параметризации), **формулировку пишет модель** (тон, длина, опечатки, путаница). Тогда покрытие обеспечено по построению, а модель добавляет то, чего не даст шаблон, - живой язык.

In [ ]:
PROMPT = """Ты помогаешь собрать набор тестовых обращений в поддержку облачного сервиса.
Напиши ОДНО обращение клиента, строго по заданному. Только текст обращения, без пояснений.

Что произошло: {reason}
Номер заказа: {order}
Сумма: {amount} руб
Прошло дней: {days}
Тон клиента: {tone}
Особенность: {twist}

Номер заказа и сумму упомяни в тексте обязательно."""

TONES = ["нейтральный", "раздраженный", "очень краткий", "многословный"]
TWISTS = ["пишет с опечатками", "задает сразу два вопроса",
          "ссылается на прошлое обращение", "путается в номере заказа"]

In [ ]:
def synth_case(slots, tone, twist):
    text = gigachat_llm.invoke(PROMPT.format(tone=tone, twist=twist, **slots)).content.strip()
    over_limit, too_late = slots["amount"] > 1000, slots["days"] > 30
    return {
        "user_input": text,
        "scenario": "возврат средств",
        "params": slots, "tone": tone, "twist": twist,
        # эталон вычисляется из слотов, а не выдумывается моделью
        "expected_answer": "Срок возврата истек" if too_late
                           else "Возврат требует подтверждения клиента" if over_limit
                           else "Возврат оформлен",
        "source": "синтез",
    }

In [ ]:
SLOT_SETS = [
    {"order": "5512", "amount": 900,   "days": 3,  "reason": "товар не подошел"},
    {"order": "7781", "amount": 4500,  "days": 20, "reason": "товар пришел поврежденным"},
    {"order": "6120", "amount": 15000, "days": 45, "reason": "передумал"},
    {"order": "4310", "amount": 2300,  "days": 10, "reason": "деньги списались дважды"},
]

slotted = [synth_case(s, TONES[i % 4], TWISTS[(i + 1) % 4])
           for i, s in enumerate(SLOT_SETS + SLOT_SETS)]
for c in slotted[:3]:
    print(f"[{c['tone']} / {c['twist']} / эталон: {c['expected_answer']}]")
    print(" ", c["user_input"].replace("\n", " ")[:220], "\n")

Сравним обе выдачи по обоим меркам сразу.

In [ ]:
slotted_texts = [c["user_input"] for c in slotted]

print("ЛЕКСИЧЕСКАЯ БЛИЗОСТЬ")
similarity_report(naive, "наивный запрос")
similarity_report(slotted_texts, "по слотам")

print("\nПОКРЫТИЕ ПРИЗНАКОВ")
coverage_report(naive, "наивный запрос")
coverage_report(slotted_texts, "по слотам")

**Пояснения к результату:**
- по лексической близости обе выдачи выглядят прилично, и наивная нередко даже лучше: модель, которой велели придумать восемь разных обращений, старается их развести словами;
- по покрытию признаков картина обратная и очень наглядная: наивная выдача покрывает **ноль признаков из пяти**. Ни одного номера заказа, ни одной суммы - зацепиться не за что, в кейсы эти тексты не превратить. В выдаче по слотам номер заказа есть во всех восьми, сумма почти во всех;
- при этом обратите внимание: признаки, которые мы задавали словами («задает два вопроса», «раздражен»), модель выполнила не везде. Заданное значением слота (номер, сумма) она проставляет надежно, заданное описанием - как получится. **Покрытие надо проверять замером, а не считать, что раз попросили - значит сделано**;
- **вывод, который стоит унести: не мерьте разнообразие синтетики косинусом.** Он меряет непохожесть формулировок. Мерить надо покрытие тех признаков, ради которых вы вообще генерировали;
- и главное про эталон. У кейсов из второй выдачи он есть - но взялся он **не от модели**, а вычислился из слотов, ровно как в параметризации. Модель отвечала только за формулировку. Это и есть рабочее разделение труда: значения и эталон - ваши, живой язык - модельный.

Температура генерации, кстати, ни на что из этого не влияет: она добавляет случайности словам, а не признакам.

## Фильтр дублей

Дубли приезжают отовсюду: из логов (клиенты пишут одинаково), из синтеза (модель однообразна), из объединения наборов. Ловятся в два прохода: точные совпадения после нормализации и близкие пары по тексту.

In [ ]:
def normalize(text):
    text = text.lower().replace("е", "е")
    text = re.sub(r"\d+", "<n>", text)          # номера заказов не должны делать кейсы разными
    text = re.sub(r"[^а-яa-z<>n ]", " ", text)
    return " ".join(text.split())

pool = naive + slotted_texts + [naive[0], "  " + naive[1].upper() + "  "]   # два дубля подложили сами
print("всего текстов:", len(pool))

seen, exact_unique = set(), []
for text in pool:
    key = normalize(text)
    if key not in seen:
        seen.add(key)
        exact_unique.append(text)
print("после снятия точных дублей:", len(exact_unique))

# близкие пары
m = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit_transform(exact_unique)  # noqa: F821
sim = cosine_similarity(m)
np.fill_diagonal(sim, 0)

THRESHOLD = 0.7
drop = set()
for i in range(len(exact_unique)):
    if i in drop:
        continue
    for j in range(i + 1, len(exact_unique)):
        if sim[i, j] > THRESHOLD:
            drop.add(j)

print(f"близких пар выше {THRESHOLD}: {int((sim > THRESHOLD).sum() // 2)} | к удалению: {len(drop)}")
print("осталось уникальных кейсов:", len(exact_unique) - len(drop))

**Пояснения к результату:**
- нормализация номеров - важная деталь: «верните деньги за заказ 5512» и «за заказ 7781» - это один кейс, а не два. Но осторожно: так можно только там, где **от числа не зависит эталон**. В параметризации выше сумма 900 и сумма 4500 давали разные эталоны, и такие кейсы схлопывать нельзя - нормализуйте только те числа, которые ни на что не влияют;
- порог 0.7 подбирается глазами по своим данным. Слишком низкий выбросит разные кейсы, слишком высокий не поймает перефразировки;
- и встречный довод: **дубли не всегда вредны**. Если вы проверяете устойчивость агента к перефразировкам, близкие пары - это и есть предмет проверки. Вредны они там, где по набору считают долю успехов: десять копий одного кейса дают ему десятикратный вес в метрике.

# Финализация датасета

## Баланс по классам и сложности

Последний шаг перед сдачей набора - посмотреть, из чего он состоит. Метрика по несбалансированному набору говорит в основном о том, чего в нем больше.

In [ ]:
# набор после объединения: первичный + параметризация + синтез
pool_rows = (
    [{"scenario": s, "difficulty": d, "source": "первичный"}
     for s, d in zip(df["scenario"], df["difficulty"])]
    + [{"scenario": c["scenario"],
        "difficulty": "сложный" if c["params"]["days"] > 30 else "средний",
        "source": "параметризация"} for c in generated]
)
pool_df = pd.DataFrame(pool_rows)

print("по классам сценариев:")
print(pool_df["scenario"].value_counts().to_string())
print("\nпо источникам:")
print(pool_df["source"].value_counts().to_string())
print("\nсочетание класс x сложность:")
print(pd.crosstab(pool_df["scenario"], pool_df["difficulty"]).to_string())

**Пояснения к результату:**
- после параметризации возвраты заняли большую часть набора: 81 кейс против 12 в каждом из остальных классов. Success Rate по такому набору - это в основном Success Rate на возвратах;
- лечится двумя способами: прореживанием разросшегося класса до сопоставимого размера **или** отказом от общей метрики в пользу метрики по каждому классу. Второе честнее: классы разной важности не обязаны быть равными по размеру, но и смешивать их в одно число тогда нельзя;
- сложные кейсы всегда в меньшинстве - их дороже придумывать. Именно на них агент и ломается, поэтому долю сложных стоит держать под присмотром отдельно.

## Типичные ошибки при сборе датасета

Собрано из того, что чаще всего портит набор уже после сдачи:

- **Набор собран под текущего агента.** Кейсы пишут, глядя на то, что агент уже умеет, - и получают проверку, которую он проходит по построению. Кейсы пишутся от задачи клиента, а не от возможностей агента.
- **Эталон, который нельзя проверить.** «Агент должен вежливо и исчерпывающе ответить» - по такому эталону не посчитать ничего. Эталон - это конкретные факты в ответе и ограничения на действия.
- **Разметка запущена без пилота.** Тысяча кейсов размечена по инструкции, в которой есть неоднозначное место, - и переразмечать придется все.
- **Согласие посчитано одним числом.** Каппа 0.42 по всему набору не говорит, что делать. Разбивка по разметчикам и по классам говорит.
- **Синтетика влита в набор без проверки.** Однообразие и отсутствие эталонов всплывают позже, когда метрика уже посчитана и озвучена.
- **Один набор на все.** Регресс (не сломалось ли то, что работало) и приемка (готовы ли выкатывать) - разные задачи с разными наборами и разными порогами.
- **Набор не обновляется.** Агент меняется, сценарии добавляются, а набор остается прошлогодним - и метрика по нему перестает что-либо значить.

## Чек-лист готовности датасета

Перед тем как считать по набору метрики и озвучивать их коллегам:

- у каждого кейса есть **состояние мира**, без которого он невоспроизводим;
- у каждого кейса есть **проверяемый эталон ответа** и **ограничения на траекторию**;
- в наборе закрыты **осмысленные** сочетания признаков, невозможные вычеркнуты руками;
- размер набора соотнесен с разницей, которую вы хотите различать, - интервал посчитан;
- данные из логов **обезличены до выгрузки**, и выгрузка согласована с тем, кто отвечает за персональные данные;
- инструкция разметчику **проверена пилотом**, согласие посчитано, проблемные места исправлены и переразмечены;
- согласие приведено **в разбивке** по разметчикам и по классам, а не одним числом;
- синтетика проверена **на покрытие признаков**, а не только на непохожесть формулировок, и у нее есть эталоны;
- дубли сняты, баланс по классам и сложности зафиксирован в отчете;
- указано, **что этот набор проверяет, а что нет**: маленький набор ловит грубые поломки, и это надо сказать вслух.

# Итоги

В этом уроке мы:

1. Разобрали состав первичного набора: схема кейса, эталон ответа и эталонная траектория в виде ограничений, покрытие сценариев и расчет нужного размера набора; посмотрели на три источника кейсов и на то, чем плох каждый.
2. Разметили кейсы сами, посчитали согласие разметчиков процентом, каппой Коэна, каппой Фляйса и альфой Криппендорфа - и увидели, где каждая из этих мерок обманывает.
3. Нашли по цифрам адрес проблемы (мягкий разметчик и класс «возврат средств»), поправили инструкцию, перезамерили - и расширили набор параметризацией и синтезом с проверкой на однообразие, дубли и баланс.

# Что запомнить

- Кейс агента - это не пара «вопрос, ответ»: без состояния мира и без ограничений на действия он невоспроизводим и непроверяем.
- Эталонная траектория - это **ограничения**, а не единственно верная последовательность: обязательные вызовы, запрещенные вызовы, частичный порядок, потолок шагов. Правильных траекторий у задачи несколько.
- Процент согласия обманывает на редких классах: 0.90 согласия при каппе 0.23 означает, что разметчики просто одинаково часто говорят «нарушения нет».
- Для шкалы 1-5 берите каппу с квадратичными весами, для номинальных признаков - без весов. Для троих и больше - каппа Фляйса, при пропусках - альфа Криппендорфа.
- Шкала «0.61-0.80 - хорошо» придумана произвольно. Порог приемки согласия - ваше решение, а не цитата.
- Одно число согласия по всему набору ничего не решает. Решает разбивка: по разметчикам находится мягкий или невнимательный, по классам - неоднозначное место инструкции.
- Систематическое расхождение - вина инструкции, а не разметчиков. Правится инструкция, спорный класс размечается заново, согласие меряется повторно.
- У синтетического кейса нет эталона. Параметризация тем и хороша, что эталон вычисляется из параметров; за синтез моделью эталоном платят отдельно.
- Разнообразие синтетики берется из явно заданных признаков, а не из температуры. Однообразие меряется до того, как набор пойдет в работу.
- Метрика по несбалансированному набору говорит о том, чего в наборе больше. Либо выравнивайте, либо считайте по классам отдельно.

# Полезные материалы

- [scikit-learn: cohen_kappa_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.cohen_kappa_score.html) - каппа Коэна, в том числе с весами.
- [statsmodels: inter-rater reliability](https://www.statsmodels.org/stable/stats.html#inter-rater-reliability) - каппа Фляйса и вспомогательные функции.
- [krippendorff](https://github.com/pln-fing-udelar/fast-krippendorff) - альфа Криппендорфа для разных шкал и пропусков.
- [Kappa paradoxes](https://pubmed.ncbi.nlm.nih.gov/2348207/) - Feinstein, Cicchetti о том, почему высокое согласие дает низкую каппу.
- [Landis, Koch (1977)](https://pubmed.ncbi.nlm.nih.gov/843571/) - та самая шкала интерпретации; полезно прочитать, что авторы называют ее произвольной.
- [τ-bench](https://arxiv.org/abs/2406.12045) - набор для оценки агентов в диалоге с пользователем и инструментами: полезен как образец описания кейса.
- [Data-Centric AI](https://datacentricai.org/) - подборка про то, почему за качество отвечают в первую очередь данные, а не модель.